# Publication snapshot v0.4.0
整理自既有研究；已移除outputs、憑證與私人路徑。見README與docs/limitations.md。
保留歷史實驗設計供檢視，不代表已修正資料切分或已在乾淨環境重跑。


# 蒸餾 Phase 4 — 合併 LoRA → 轉 GGUF → 量化 → Ollama 部署驗收

流程：①把 LoRA adapter 合併回 base model（存成一般 fp16 checkpoint）→ ②用 llama.cpp 的 `convert_hf_to_gguf.py` 轉成 F16 GGUF（純 Python，不用編譯）→ ③用 `ollama create --quantize q4_K_M` 直接讓 Ollama 幫你量化（不用自己編 `llama-quantize`，Windows 上省一大堆麻煩）→ ④分別把「合併後 fp16」跟「量化後 q4_K_M」用 Ollama 跑起來，在同一批資料上比準度/延遲/檔案大小，量出「量化到底掉多少」。

**這本一樣要在你的電腦（GPU + 已裝好 ollama）上跑。** 跑完把結果貼給我分析。


## 0. 設定

In [ ]:
import os, sys, json, time, subprocess, shutil
print(sys.executable)

STUDENT_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
ADAPTER_DIR   = "student_halluc_detector"   # Phase 2 存 adapter 的地方
MERGED_DIR    = "student_merged_fp16"             # 合併後的一般 HF checkpoint
LLAMACPP_DIR  = "llama.cpp"                        # llama.cpp 原始碼要 clone 到這裡
GGUF_F16      = "student-f16.gguf"
OLLAMA_NAME_F16 = "student-halluc-f16"
OLLAMA_NAME_Q4  = "student-halluc-q4km"

SYS_STUDENT = "You are a strict factual-consistency checker. Given a SOURCE article and a SUMMARY, answer with exactly one word: CONSISTENT or HALLUCINATED."

def run_cmd(cmd, env=None):
    """統一的 subprocess 執行方式：強制子行程用 UTF-8 輸出、父行程用 UTF-8(容錯) 解碼，
    避免路徑含中文字（面試/論文深化研究）在 Windows 下用系統內碼（cp950）印字，
    被 Jupyter/subprocess 用 UTF-8 解碼時整個 crash（UnicodeDecodeError）。"""
    e = os.environ.copy()
    e["PYTHONUTF8"] = "1"
    e["PYTHONIOENCODING"] = "utf-8"
    if env:
        e.update(env)
    r = subprocess.run(cmd, env=e, capture_output=True, text=True,
                       encoding="utf-8", errors="replace")
    if r.stdout:
        print(r.stdout)
    if r.stderr:
        print(r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"指令失敗（exit {r.returncode}）: {' '.join(map(str, cmd))}")
    return r


## 1. 合併 LoRA adapter 回 base model（存成一般 fp16 checkpoint）\n\n注意：合併要用**沒有 4bit 量化**的 base model 載入（fp16/bf16），不能直接對 bnb 4bit 模型做 `merge_and_unload()`（數值上不乾淨）。Qwen3-4B 在 bf16 下大概 8GB，12GB 卡跑得動，但建議先重啟 kernel、確保沒有其他 Phase 2/3 留下的模型還佔著顯存。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# 注意：直接從原始 STUDENT_MODEL 載入 tokenizer，不要用 ADAPTER_DIR。
# LoRA 訓練不會動到 tokenizer 內容，兩邊詞表完全一樣。
tok = AutoTokenizer.from_pretrained(STUDENT_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(STUDENT_MODEL, torch_dtype=torch.bfloat16, device_map="auto")
merged = PeftModel.from_pretrained(base, ADAPTER_DIR)
merged = merged.merge_and_unload()   # LoRA 權重疊回底模，變回一般稠密模型

os.makedirs(MERGED_DIR, exist_ok=True)
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tok.save_pretrained(MERGED_DIR)
print("✓ 合併完成，存在", MERGED_DIR)

# --- 修正已知 bug：save_pretrained 存出來的 tokenizer_config.json 裡
# extra_special_tokens 有時會被寫成 []（list），但這版 transformers 載入時
# 預期是 {}（dict），會讓任何之後要重新讀這個資料夾的工具（包括 llama.cpp
# 的轉檔腳本）直接 AttributeError 崩潰。存完就地修正，一次到位。 ---
tok_cfg_path = os.path.join(MERGED_DIR, "tokenizer_config.json")
with open(tok_cfg_path, encoding="utf-8") as f:
    tok_cfg = json.load(f)
if isinstance(tok_cfg.get("extra_special_tokens"), list):
    tok_cfg["extra_special_tokens"] = {}
    with open(tok_cfg_path, "w", encoding="utf-8") as f:
        json.dump(tok_cfg, f, ensure_ascii=False, indent=2)
    print("✓ 已修正 tokenizer_config.json 的 extra_special_tokens 欄位（[] → {}）")
else:
    print("· tokenizer_config.json 的 extra_special_tokens 欄位正常，不需修正")

# 檔案大小（之後跟 GGUF 版本比壓縮率用）
def dir_size_mb(d):
    return sum(os.path.getsize(os.path.join(d, f)) for f in os.listdir(d)
               if os.path.isfile(os.path.join(d, f))) / 1024**2
print(f"合併後 fp16 checkpoint 大小：約 {dir_size_mb(MERGED_DIR):.0f} MB")


## 2. 拿 llama.cpp 的轉檔腳本（純 Python，不用編譯）

In [ ]:
if not os.path.isdir(LLAMACPP_DIR):
    run_cmd(["git", "clone", "--depth", "1",
             "https://github.com/ggml-org/llama.cpp.git", LLAMACPP_DIR])
else:
    print("llama.cpp 已存在，略過 clone（想更新就手動 git pull）")

run_cmd([sys.executable, "-m", "pip", "install", "-q",
         "-r", os.path.join(LLAMACPP_DIR, "requirements.txt")])
print("✓ 轉檔相依套件裝好了")


## 3. 轉成 F16 GGUF

In [ ]:
import json, os
tok_cfg_path = os.path.join("student_merged_fp16", "tokenizer_config.json")
with open(tok_cfg_path, encoding="utf-8") as f:
    cfg = json.load(f)
if isinstance(cfg.get("extra_special_tokens"), list):
    cfg["extra_special_tokens"] = {}
    with open(tok_cfg_path, "w", encoding="utf-8") as f:
        json.dump(cfg, f, ensure_ascii=False, indent=2)
    print("已修正")
else:
    print("欄位正常")

In [ ]:
convert_script = os.path.join(LLAMACPP_DIR, "convert_hf_to_gguf.py")
run_cmd([sys.executable, convert_script, MERGED_DIR,
         "--outfile", GGUF_F16, "--outtype", "f16"])
print("✓ 轉檔完成：", GGUF_F16, f"| 大小約 {os.path.getsize(GGUF_F16)/1024**2:.0f} MB")


## 4. 用 Ollama 建立兩個版本：F16（未量化，對照組）跟 Q4_K_M（量化）

不用自己編 `llama-quantize`（Windows 上编譯很麻煩）—— 現在的 Ollama 支援直接吃 F16 GGUF、用 `--quantize` 幫你量化。


In [ ]:
modelfile = f"""FROM ./{GGUF_F16}
SYSTEM \"\"\"{SYS_STUDENT}\"\"\"
PARAMETER temperature 0
PARAMETER num_ctx 4096
"""
with open("Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile)
print(modelfile)


In [ ]:
# 對照組：F16、不量化
run_cmd(["ollama", "create", OLLAMA_NAME_F16, "-f", "Modelfile"])
print(f"✓ 已建立 {OLLAMA_NAME_F16}（F16，未量化）")


In [ ]:
# 量化版：Q4_K_M
run_cmd(["ollama", "create", "--quantize", "q4_K_M", OLLAMA_NAME_Q4, "-f", "Modelfile"])
print(f"✓ 已建立 {OLLAMA_NAME_Q4}（Q4_K_M，量化後）")

# 列出兩個模型的實際檔案大小（ollama list 會印出磁碟佔用）
run_cmd(["ollama", "list"])


## 5. 驗收：F16 vs Q4_K_M vs 原本 transformers+bnb 版，準度/延遲一起比

用跟 Phase 3 / 實驗1 一樣的評估邏輯（同一份 FaithBench 750 筆，或至少 Phase 3 的 historical validation split (not an independent test)），
分別呼叫這兩個 ollama 模型跑一輪，量「量化前後掉多少準度、換來多少速度」。


In [ ]:
import ollama as ollama_client

def judge_via_ollama(model_name, source, summary, max_retries=1):
    prompt = f"SOURCE:\n{source}\n\nSUMMARY:\n{summary}\n\nIs the summary factually consistent with the source? Answer CONSISTENT or HALLUCINATED."
    t0 = time.time()
    resp = ollama_client.chat(model=model_name, stream=False,
                              messages=[{"role": "user", "content": prompt}])
    dt = time.time() - t0
    text = resp["message"]["content"].upper()
    pred = 1 if "HALLUCINAT" in text else 0
    return pred, dt, text.strip()

# sanity check 一筆（source/summary 換成你自己資料裡的一筆測試）
# pred, dt, raw = judge_via_ollama(OLLAMA_NAME_Q4, test_source, test_summary)
# print(pred, dt, raw)


In [ ]:
# 建議直接沿用 Phase 3 重建的 historical validation split (not an independent test)（按列切分；同來源文章可能重疊），或直接拿 FaithBench 750 筆重跑一次。
# 這裡示範用 Phase 3 的 test_set（先跑過 Phase 3 那本、或在這裡重新載入 distill_data.jsonl 切一樣的 split）。
from datasets import Dataset
DATA_JSONL = "distill_data.jsonl"
SEED = 42
rows = []
with open(DATA_JSONL, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if r.get("teacher_label") in (0, 1):
            rows.append({"source": r["source"], "summary": r["summary"], "label": int(r["teacher_label"])})
ds_all = Dataset.from_list(rows)
test_set = ds_all.train_test_split(test_size=0.15, seed=SEED)["test"]
print("test set 筆數：", len(test_set))

results = {}
for model_name in [OLLAMA_NAME_F16, OLLAMA_NAME_Q4]:
    preds, labels, latencies = [], [], []
    for i in range(len(test_set)):
        ex = test_set[i]
        pred, dt, _ = judge_via_ollama(model_name, ex["source"], ex["summary"])
        preds.append(pred); labels.append(ex["label"]); latencies.append(dt)
        if (i+1) % 50 == 0:
            print(f"  {model_name}: {i+1}/{len(test_set)}")
    results[model_name] = {"preds": preds, "labels": labels, "latencies": latencies}

from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
for name, r in results.items():
    acc = accuracy_score(r["labels"], r["preds"])
    bacc = balanced_accuracy_score(r["labels"], r["preds"])
    avg_lat = sum(r["latencies"]) / len(r["latencies"])
    print(f"\n=== {name} ===")
    print("accuracy:", round(acc,3), "| balanced_accuracy:", round(bacc,3), "| 平均延遲:", round(avg_lat,3), "秒")
    print("confusion matrix:\n", confusion_matrix(r["labels"], r["preds"]))


### 5.1 診斷：recall 偏低是不是因為 thinking-mode 意外開啟、輸出被截斷成空

檢查幾筆「真的是 HALLUCINATED、但模型判成 CONSISTENT」（false negative）的原始輸出，
看有沒有 `<think>` 標籤、輸出異常短/空——如果有，代表 chat template 誤觸發推理模式，
答案還沒生成就被截斷，字串比對抓不到 "HALLUCINAT" 就預設判 0。
排除這個假設之後，才能安心把 recall/specificity 落差歸因於量化數值誤差。


In [ ]:
import re

def inspect_raw(model_name, n=8):
    shown = 0
    for i in range(len(test_set)):
        if shown >= n:
            break
        ex = test_set[i]
        if ex["label"] != 1:   # 只看真的是 HALLUCINATED 的樣本
            continue
        pred, dt, raw = judge_via_ollama(model_name, ex["source"], ex["summary"])
        if pred == 0:   # 模型誤判成 CONSISTENT（false negative）
            has_think = bool(re.search(r"<think>", raw, re.I))
            print(f"[{model_name}] id={i} pred={pred} label={ex['label']} 含<think>={has_think} 原始輸出長度={len(raw)}")
            print("  raw:", raw[:200].replace(chr(10), " "))
            shown += 1
    if shown == 0:
        print(f"[{model_name}] 抽樣沒抽到 false negative，把 n 調大一點再跑")

print("=== F16 false negative 樣本檢查 ===")
inspect_raw(OLLAMA_NAME_F16)
print()
print("=== Q4_K_M false negative 樣本檢查 ===")
inspect_raw(OLLAMA_NAME_Q4)

print()
print("判讀：如果 raw 輸出裡出現 <think> 標籤、或輸出長度異常短/空，代表 thinking-mode 意外觸發、答案被截斷——")
print("這樣要在 Modelfile 加 `/no_think`（或確認 chat template 沒有誤觸發推理模式）再重跑。")
print("如果都是乾淨的 'CONSISTENT' 這種正常輸出，就排除這個假設，落回量化數值誤差的解釋。")


## 6. 檔案大小/壓縮率總表

In [ ]:
import os
def fsize_mb(p):
    return os.path.getsize(p)/1024**2 if os.path.isfile(p) else None

print("合併後 fp16 (safetensors 資料夾)：約", f"{sum(os.path.getsize(os.path.join(MERGED_DIR,f)) for f in os.listdir(MERGED_DIR) if os.path.isfile(os.path.join(MERGED_DIR,f)))/1024**2:.0f} MB")
print("F16 GGUF：", f"{fsize_mb(GGUF_F16):.0f} MB" if fsize_mb(GGUF_F16) else "找不到檔案")
print("\n(量化後 Q4_K_M 的實際大小看上面 `ollama list` 的輸出那一欄)")
print("把這三個數字 + 上面的準度/延遲表一起貼給我，我幫你整理成最終的『準度-成本』對照表。")


In [ ]:
# !dir student_merged_fp16

## 7. 接下來

跑完這本，你會有一張完整的表：`transformers+bnb 4bit`（Phase 3 量過：0.568 balanced acc / 1.36s） vs `ollama F16`（無量化對照組） vs `ollama Q4_K_M`（真正要部署的版本），三者的準度、延遲、檔案大小都有數字。

這張表 + FaithBench 750 筆的完整偵測器比較表，就是整個專案收尾要放進論文/履歷的核心證據。把這裡跑出來的結果貼給我，我幫你一起看「量化後掉了多少準度、換到多少速度/體積」，並幫你把最終的準度-成本圖和結論段落定下來。
